# Solar Panel Fault Detection — Universal Data Pipeline
# Capacity-Agnostic Version (works on any station worldwide)
# All absolute power/current/voltage features are normalized by station capacity.
# Models trained on this dataset can be applied to ANY solar station.

In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
import subprocess, sys

REQUIRED_PACKAGES = [
    'pandas', 'numpy', 'matplotlib', 'seaborn', 'scikit-learn',
    'xgboost', 'lightgbm', 'catboost', 'joblib', 'tqdm',
    'pvlib', 'requests',
]

def install_missing_packages():
    for pkg in REQUIRED_PACKAGES:
        pkg_import = pkg.replace('-', '_')
        if pkg == 'scikit-learn': pkg_import = 'sklearn'
        try:
            __import__(pkg_import)
        except ImportError:
            print(f"[INSTALL] Installing {pkg}...")
            subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

install_missing_packages()


[INSTALL] Installing catboost...
[INSTALL] Installing pvlib...


In [3]:
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import pvlib
from pvlib.pvsystem import PVSystem, Array, FixedMount
from pvlib.location import Location
import os, json, glob
from collections import Counter
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
print('All libraries imported successfully')


All libraries imported successfully


In [4]:
# ================================================================
# CONFIGURATION
# Adjust STATION_ID and DC_CAPACITY_KW for your target station.
# All features are normalized by capacity so models stay universal.
# ================================================================
CONFIG = {
    'STATION_ID'       : 2107,
    'STATION_NAME'     : 'Arbuckle CA',
    'DC_CAPACITY_KW'   : 893,
    'NUM_INVERTERS'    : 24,
    'INVERTER_EFF'     : 0.96,
    'TEMP_COEFF_POWER' : -0.004,
    'REF_TEMP'         : 25,
    'DC_CAPACITY_W'    : 893 * 1000,
    'MAX_DC_POWER'     : 893 * 1000 * 2,
    'DATA_DIR'         : '/content/drive/MyDrive/SolarGard4/solar_data',
    'MODEL_DIR'        : '/content/drive/MyDrive/SolarGard4/solar_models_universal',
    'FAULT_LABELS'     : {
        0: 'Normal',            1: 'Partial Shading',
        2: 'Soiling',           3: 'Degradation',
        4: 'Inverter Fault',    5: 'Open-Circuit String',
        6: 'Short-Circuit',     7: 'Sensor Fault',
    },
    # 27 universal (normalized/relative) features — same count, no absolute power
    'NUM_FEATURES'     : 27,
}

print('=' * 60)
print('  SOLAR PANEL FAULT DETECTION — UNIVERSAL DATA PIPELINE')
print('=' * 60)
print(f"  Station:      PVDAQ {CONFIG['STATION_ID']} - {CONFIG['STATION_NAME']}")
print(f"  DC Capacity:  {CONFIG['DC_CAPACITY_KW']} kW")
print(f"  Inverters:    {CONFIG['NUM_INVERTERS']}")
print(f"  Feature set:  NORMALIZED (capacity-agnostic)")
print(f"  Fault Types:  {len(CONFIG['FAULT_LABELS'])} (F0-F7)")
print('=' * 60)
os.makedirs(CONFIG['DATA_DIR'], exist_ok=True)
os.makedirs(CONFIG['MODEL_DIR'], exist_ok=True)


  SOLAR PANEL FAULT DETECTION — UNIVERSAL DATA PIPELINE
  Station:      PVDAQ 2107 - Arbuckle CA
  DC Capacity:  893 kW
  Inverters:    24
  Feature set:  NORMALIZED (capacity-agnostic)
  Fault Types:  8 (F0-F7)


## Step 1 — Download Raw PVDAQ Data

In [5]:
import urllib.request

def download_solar_data(config):
    data_dir   = config['DATA_DIR']
    station_id = config['STATION_ID']
    s3_base    = (f"https://oedi-data-lake.s3.amazonaws.com/pvdaq/"
                  f"2023-solar-data-prize/{station_id}_OEDI/data")
    files_to_download = {
        'electrical' : f"{s3_base}/{station_id}_electrical_data.csv",
        'environment': f"{s3_base}/{station_id}_environment_data.csv",
        'irradiance' : f"{s3_base}/{station_id}_irradiance_data.csv",
        'meter'      : f"{s3_base}/{station_id}_meter_15m_data.csv",
    }
    loaded_data = {}
    for name, url in files_to_download.items():
        local_path = os.path.join(data_dir, f"{name}_data.csv")
        if os.path.exists(local_path):
            print(f"  [SKIP] {name}: already exists")
        else:
            print(f"  [DOWNLOAD] {name}...")
            try:
                urllib.request.urlretrieve(url, local_path)
                print(f"    -> Saved {os.path.getsize(local_path):,} bytes")
            except Exception as e:
                print(f"    -> Download failed: {e}")
                continue
        try:
            df = pd.read_csv(local_path)
            loaded_data[name] = df
            print(f"    -> Loaded {len(df):,} rows, {len(df.columns)} columns")
        except Exception as e:
            print(f"    -> Error reading CSV: {e}")
    return loaded_data

loaded_data = download_solar_data(CONFIG)


  [DOWNLOAD] electrical...
    -> Saved 438,569,900 bytes
    -> Loaded 632,952 rows, 120 columns
  [DOWNLOAD] environment...
    -> Saved 7,700,486 bytes
    -> Loaded 206,008 rows, 4 columns
  [DOWNLOAD] irradiance...
    -> Saved 13,990,782 bytes
    -> Loaded 531,019 rows, 2 columns
  [DOWNLOAD] meter...
    -> Saved 6,319,525 bytes
    -> Loaded 240,062 rows, 2 columns


## Step 2 — Clean & Process Electrical Data

In [6]:
def _find_timestamp_column(df, name="data"):
    for c in df.columns:
        cl = c.lower().strip()
        if any(k in cl for k in ["date", "time", "timestamp", "datetime", "measured_on"]):
            return c
    print(f"    [WARN] No timestamp column in {name}, using first column: '{df.columns[0]}'")
    return df.columns[0]


def _clean_electrical_data(df, config):
    """Replace extreme sensor glitch values with NaN before any computation."""
    cleaned = 0
    for i in range(1, config["NUM_INVERTERS"] + 1):
        limits = {
            "dc_current": 200,  "dc_voltage": 1500,
            "ac_current": 500,  "ac_voltage": 1500,
            "ac_power"  : 200,
        }
        for metric, limit in limits.items():
            cols = [c for c in df.columns if f"inv_{i:02d}_{metric}" in c]
            for c in cols:
                extreme = df[c] > limit
                if extreme.any():
                    n = extreme.sum()
                    print(f"    [CLEAN] Inv {i:02d} {metric}: {n} values > {limit} (max={df[c].max():.1f}) -> NaN")
                    df.loc[extreme, c] = np.nan
                    cleaned += n
    if cleaned > 0:
        print(f"    [CLEAN] Total corrupted values replaced: {cleaned}")
    return df


def _process_electrical_data(df, config):
    dc_cap_w = config["DC_CAPACITY_W"]
    max_dc   = config["MAX_DC_POWER"]
    inv_eff  = config["INVERTER_EFF"]

    df = _clean_electrical_data(df, config)

    dc_power_cols = []
    for i in range(1, config["NUM_INVERTERS"] + 1):
        curr_cols   = [c for c in df.columns if f"inv_{i:02d}_dc_current" in c]
        volt_cols   = [c for c in df.columns if f"inv_{i:02d}_dc_voltage" in c]
        ac_pwr_cols = [c for c in df.columns if f"inv_{i:02d}_ac_power"   in c]

        if curr_cols and volt_cols:
            df[f"DC_POWER_INV{i}"] = df[curr_cols[0]] * df[volt_cols[0]]
            dc_power_cols.append(f"DC_POWER_INV{i}")
        elif curr_cols and ac_pwr_cols:
            print(f"    [FIX] Inv {i:02d}: missing dc_voltage, estimating DC from AC (kW->W)")
            df[f"DC_POWER_INV{i}"] = df[ac_pwr_cols[0]] * 1000 / inv_eff
            dc_power_cols.append(f"DC_POWER_INV{i}")
        else:
            print(f"    [WARN] Inv {i:02d}: cannot compute DC_POWER")

    if dc_power_cols:
        df[dc_power_cols] = df[dc_power_cols].fillna(0)
        df["DC_POWER"]    = df[dc_power_cols].sum(axis=1)
        daytime_dc        = df.loc[df["DC_POWER"] > 0, "DC_POWER"]
        print(f"    [OK] DC_POWER: mean={daytime_dc.mean():.0f} W, max={daytime_dc.max():.0f} W")
    else:
        df["DC_POWER"] = 0

    ac_power_cols = [c for c in df.columns if "ac_power" in c.lower() and "inv_" in c.lower()]
    if ac_power_cols:
        df["AC_POWER"] = df[ac_power_cols].fillna(0).sum(axis=1)

    if "AC_POWER" in df.columns:
        ac_max = df["AC_POWER"].max()
        threshold = dc_cap_w * 0.5
        if ac_max < threshold and ac_max > 0:
            print(f"    [FIX] AC_POWER in kW (max={ac_max:.1f}), converting to W")
            df["AC_POWER"] = df["AC_POWER"] * 1000

    if "DC_POWER" in df.columns:
        before_cap = (df["DC_POWER"] > max_dc).sum()
        df.loc[df["DC_POWER"] > max_dc, "DC_POWER"] = np.nan
        if before_cap > 0:
            print(f"    [FIX] DC_POWER capped: {before_cap:,} values -> NaN")

    return df


df_elec = loaded_data.get("electrical", pd.DataFrame()).copy()
print(f"Processing electrical data ({len(df_elec):,} rows)...")
df_elec = _process_electrical_data(df_elec, CONFIG)
elec_save_path = os.path.join(CONFIG["DATA_DIR"], "electrical_processed.csv")
df_elec.to_csv(elec_save_path, index=False)
print(f"Saved: {elec_save_path}")


Processing electrical data (632,952 rows)...
    [CLEAN] Inv 02 dc_voltage: 1 values > 1500 (max=1750.8) -> NaN
    [CLEAN] Inv 04 ac_current: 1 values > 500 (max=93500000000.0) -> NaN
    [CLEAN] Inv 06 dc_current: 1 values > 200 (max=45600000000.0) -> NaN
    [CLEAN] Inv 07 dc_voltage: 1 values > 1500 (max=1380000000000.0) -> NaN
    [CLEAN] Total corrupted values replaced: 4
    [FIX] Inv 05: missing dc_voltage, estimating DC from AC (kW->W)
    [OK] DC_POWER: mean=309161 W, max=742943 W
    [FIX] AC_POWER in kW (max=721.6), converting to W
Saved: /content/drive/MyDrive/SolarGard4/solar_data/electrical_processed.csv


## Step 3 — Merge All Sources

In [7]:
def merge_all_data(loaded_data, df_elec, config):
    df_env   = loaded_data.get("environment", pd.DataFrame()).copy()
    df_irr   = loaded_data.get("irradiance",  pd.DataFrame()).copy()
    df_meter = loaded_data.get("meter",        pd.DataFrame()).copy()

    env_map = {
        "ambient_temperature_o_149575": "AMBIENT_TEMP",
        "wind_speed_o_149576"         : "WIND_SPEED",
        "wind_direction_o_149577"     : "WIND_DIRECTION",
    }
    for old, new in env_map.items():
        if old in df_env.columns:
            df_env.rename(columns={old: new}, inplace=True)

    if "AMBIENT_TEMP" in df_env.columns and df_env["AMBIENT_TEMP"].max() > 60:
        print(f"    [FIX] Temperature in Fahrenheit, converting to Celsius")
        df_env["AMBIENT_TEMP"] = (df_env["AMBIENT_TEMP"] - 32) * 5 / 9

    if "poa_irradiance_o_149574" in df_irr.columns:
        df_irr.rename(columns={"poa_irradiance_o_149574": "POA_IRRADIANCE"}, inplace=True)

    if "meter_revenue_grade_ac_output_meter_149578" in df_meter.columns:
        df_meter.rename(columns={"meter_revenue_grade_ac_output_meter_149578": "METER_AC_POWER"}, inplace=True)

    dfs_to_merge = []
    for name, df in [("electrical", df_elec), ("environment", df_env),
                     ("irradiance", df_irr), ("meter", df_meter)]:
        if len(df) == 0:
            print(f"    [SKIP] {name}: empty")
            continue
        time_col = _find_timestamp_column(df, name)
        df["timestamp"] = pd.to_datetime(df[time_col], errors="coerce")
        df = df.dropna(subset=["timestamp"])
        if time_col != "timestamp":
            df = df.drop(columns=[time_col], errors="ignore")
        dfs_to_merge.append((name, df))
        print(f"    [OK] {name}: {len(df):,} rows")

    merged = dfs_to_merge[0][1]
    for i in range(1, len(dfs_to_merge)):
        name, df = dfs_to_merge[i]
        before   = len(merged)
        merged   = pd.merge(merged, df, on="timestamp", how="inner",
                            suffixes=("", f"_{name}"))
        print(f"    Merge with {name}: {before:,} -> {len(merged):,} rows")

    merged = merged.sort_values("timestamp").reset_index(drop=True)
    merged = merged.loc[:, ~merged.columns.duplicated()]
    merged = merged.dropna(axis=1, how="all")
    print(f"  Final merged: {len(merged):,} rows x {len(merged.columns)} columns")
    return merged


df_raw = merge_all_data(loaded_data, df_elec, CONFIG)
merged_save_path = os.path.join(CONFIG["DATA_DIR"], "merged_dataset.csv")
df_raw.to_csv(merged_save_path, index=False)
print(f"Saved: {merged_save_path}")


    [FIX] Temperature in Fahrenheit, converting to Celsius
    [OK] electrical: 632,952 rows
    [OK] environment: 206,008 rows
    [OK] irradiance: 531,019 rows
    [OK] meter: 240,062 rows
    Merge with environment: 632,952 -> 206,008 rows
    Merge with irradiance: 206,008 -> 175,566 rows
    Merge with meter: 175,566 -> 175,320 rows
  Final merged: 175,320 rows x 151 columns
Saved: /content/drive/MyDrive/SolarGard4/solar_data/merged_dataset.csv


:## Step 4 — Feature Engineering (Universal / Normalized)

In [8]:
# ================================================================
# UNIVERSAL FEATURE ENGINEERING
# ================================================================
# KEY PRINCIPLE: Every feature that depends on station capacity
# is NORMALIZED by dividing by DC_CAPACITY_W (or a derived value).
# This makes the feature dimensionless (0-1 range) and universally
# comparable across any station in the world.
#
# Features kept as-is (already universal):
#   IRRADIATION, AMBIENT_TEMP, MODULE_TEMP, temperatures,
#   solar geometry angles, ratios (EFFICIENCY, PR, DC_AC_RATIO),
#   time features (HOUR, DAY_OF_YEAR, MONTH)
#
# Features REPLACED by normalized versions:
#   DC_POWER        -> NORM_DC_POWER    (DC_POWER / DC_CAPACITY_W)
#   AC_POWER        -> NORM_AC_POWER    (AC_POWER / AC_CAPACITY_W)
#   DC_CURRENT      -> NORM_DC_CURRENT  (DC_CURRENT / RATED_DC_CURRENT)
#   DC_VOLTAGE      -> NORM_DC_VOLTAGE  (DC_VOLTAGE / RATED_DC_VOLTAGE)
#   POWER_VOLATILITY-> NORM_POWER_VOLATILITY
#   CURRENT_IMBALANCE-> NORM_CURRENT_IMBALANCE
#   VOLTAGE_SPREAD  -> NORM_VOLTAGE_SPREAD
# ================================================================

def _build_universal_features(df, config):
    """Engineer 29 NORMALIZED features — capacity-agnostic."""
    dc_cap_w    = config["DC_CAPACITY_W"]
    inv_eff     = config["INVERTER_EFF"]
    temp_coeff  = config["TEMP_COEFF_POWER"]
    ref_temp    = config["REF_TEMP"]
    num_inv     = config["NUM_INVERTERS"]

    # ── Time features (universal) ────────────────────────────
    df["HOUR"]       = df["timestamp"].dt.hour
    df["DAY_OF_YEAR"]= df["timestamp"].dt.dayofyear
    df["MONTH"]      = df["timestamp"].dt.month

    # ── Irradiation (universal) ───────────────────────────────
    if "IRRADIATION" not in df.columns or df["IRRADIATION"].sum() == 0:
        for alt in ["POA_IRRADIANCE", "GHI", "DNI", "DHI"]:
            if alt in df.columns:
                df["IRRADIATION"] = df[alt]
                print(f"    [FIX] IRRADIATION mapped from {alt}")
                break
        else:
            df["IRRADIATION"] = 0
            print("    [WARN] No irradiance column found")

    # ── Ambient Temperature (universal) ──────────────────────
    if "AMBIENT_TEMP" not in df.columns:
        for alt in ["ambient_temperature_o_149575", "Ambient_Temp", "temp_air"]:
            if alt in df.columns:
                df["AMBIENT_TEMP"] = df[alt]
                if df["AMBIENT_TEMP"].max() > 60:
                    df["AMBIENT_TEMP"] = (df["AMBIENT_TEMP"] - 32) * 5 / 9
                break
        else:
            df["AMBIENT_TEMP"] = 25

    # ── Wind Speed ───────────────────────────────────────────
    if "WIND_SPEED" not in df.columns:
        for alt in ["wind_speed_o_149576", "Wind_Speed"]:
            if alt in df.columns:
                df["WIND_SPEED"] = df[alt]
                break
        else:
            df["WIND_SPEED"] = 3

    # ── Module Temperature via Faiman model ──────────────────
    if "MODULE_TEMP" not in df.columns or df["MODULE_TEMP"].nunique() <= 1:
        a_faiman, b_faiman = -3.56, -0.075
        df["MODULE_TEMP"] = (df["AMBIENT_TEMP"]
                             + df["IRRADIATION"]
                             * np.exp(a_faiman + b_faiman * df["WIND_SPEED"].clip(lower=0.1)))
        print(f"    [OK] MODULE_TEMP via Faiman model: mean={df['MODULE_TEMP'].mean():.1f}°C")

    # ── Rated DC voltage & current per system ────────────────
    rated_dc_voltage  = num_inv * 480          # typical string voltage
    rated_dc_current  = dc_cap_w / rated_dc_voltage

    # ── Raw DC / AC voltage & current (intermediate) ─────────
    inv_dc_volt_cols = [c for c in df.columns if "dc_voltage" in c.lower() and "inv_" in c.lower()]
    if inv_dc_volt_cols:
        df["DC_VOLTAGE"] = df[inv_dc_volt_cols].mean(axis=1)
    else:
        df["DC_VOLTAGE"] = rated_dc_voltage

    df["DC_CURRENT"]  = np.where(df["DC_VOLTAGE"] > 0, df["DC_POWER"] / df["DC_VOLTAGE"], 0)
    df["AC_CURRENT"]  = np.where(df["AC_POWER"]   > 0, df["AC_POWER"] / (480 * np.sqrt(3)), 0)

    # ── Ratio & efficiency features (already universal) ──────
    df["DC_AC_RATIO"]       = np.where(df["AC_POWER"] > 0, df["DC_POWER"] / df["AC_POWER"], 0)
    df["EFFICIENCY"]        = np.where(df["IRRADIATION"] > 0,
                                        (df["DC_POWER"] / (df["IRRADIATION"] * dc_cap_w / 1000)) * 100, 0)
    df["PERFORMANCE_RATIO"] = np.where(df["IRRADIATION"] > 0,
                                        (df["AC_POWER"] / (df["IRRADIATION"] / 1000 * dc_cap_w * inv_eff)) * 100, 0)

    # ── Temperature features (universal) ─────────────────────
    df["TEMP_COEFF"]      = 1 + temp_coeff * (df["MODULE_TEMP"] - ref_temp)
    df["TEMP_DIFFERENCE"] = df["MODULE_TEMP"] - df["AMBIENT_TEMP"]
    df["TEMP_DEV"]        = df["MODULE_TEMP"] - (df["AMBIENT_TEMP"] + (df["IRRADIATION"] / 800) * 30)

    # ── Solar geometry (universal) ────────────────────────────
    df["ZENITH_ANGLE"]  = np.clip(90 - (90 * np.cos((df["HOUR"] - 12) * np.pi / 12)), 0, 90)
    df["AZIMUTH_ANGLE"] = np.clip(180 + (df["HOUR"] - 12) * 15, 0, 360)

    df["CLEARNESS_INDEX"] = np.clip(
        df["IRRADIATION"] / (1000 * np.cos(df["ZENITH_ANGLE"] * np.pi / 180) + 0.01), 0, 1.5)
    df["DIFFUSE_RATIO"]   = np.clip(1 - df["CLEARNESS_INDEX"] * 0.8, 0, 1)

    df["CLOUD_COVER_EST"] = np.clip((1 - df["CLEARNESS_INDEX"]) * 100, 0, 100)
    df["SKY_TEMP_EST"]    = df["AMBIENT_TEMP"] - 20 - df["CLOUD_COVER_EST"] * 0.15
    df["DC_AC_DEV"]       = df["DC_AC_RATIO"] - (1 / inv_eff)

    # ================================================================
    # NORMALIZED FEATURES (replacing absolute counterparts)
    # ================================================================
    ac_cap_w = dc_cap_w * inv_eff

    # NORM_DC_POWER: 0 = no production, 1 = full rated capacity
    df["NORM_DC_POWER"]        = np.clip(df["DC_POWER"] / dc_cap_w, 0, 1.5)

    # NORM_AC_POWER: 0 = no output, 1 = full rated AC output
    df["NORM_AC_POWER"]        = np.clip(df["AC_POWER"] / ac_cap_w, 0, 1.5)

    # NORM_DC_VOLTAGE: fraction of rated string voltage
    df["NORM_DC_VOLTAGE"]      = np.clip(df["DC_VOLTAGE"] / rated_dc_voltage, 0, 2.0)

    # NORM_DC_CURRENT: fraction of rated DC current
    df["NORM_DC_CURRENT"]      = np.clip(
        np.where(dc_cap_w > 0, df["DC_CURRENT"] / rated_dc_current, 0), 0, 2.0)

    # NORM_POWER_VOLATILITY: step-to-step power change as fraction of capacity
    df["NORM_POWER_VOLATILITY"]= np.clip(
        np.abs(df["DC_POWER"].diff().fillna(0)) / dc_cap_w, 0, 1.0)

    # NORM_CURRENT_IMBALANCE: imbalance as fraction of rated DC current
    rolling_curr = df["DC_CURRENT"].rolling(6, min_periods=1).mean()
    df["NORM_CURRENT_IMBALANCE"] = np.clip(
        np.abs(df["DC_CURRENT"] - rolling_curr) / (rated_dc_current + 1e-6), 0, 1.0)

    # NORM_VOLTAGE_SPREAD: voltage deviation as fraction of rated voltage
    rolling_volt = df["DC_VOLTAGE"].rolling(6, min_periods=1).mean()
    df["NORM_VOLTAGE_SPREAD"]  = np.clip(
        np.abs(df["DC_VOLTAGE"] - rolling_volt) / (rated_dc_voltage + 1e-6), 0, 1.0)

    return df


def preprocess_and_engineer_universal(df, config):
    """Full universal preprocessing pipeline."""
    if len(df) == 0:
        print("  [ERROR] Empty input dataframe")
        return pd.DataFrame(), []

    print(f"  Starting with {len(df):,} rows")

    # Night filter (irradiance < 50 W/m²)
    irr_col = None
    for c in ["POA_IRRADIANCE", "IRRADIATION"]:
        if c in df.columns:
            irr_col = c
            break

    if irr_col:
        night_mask  = df[irr_col] < 50
        night_count = night_mask.sum()
        df = df[~night_mask].copy()
        print(f"  Night-drop: removed {night_count:,} rows ({irr_col} < 50 W/m²)")

    print("  Engineering universal features...")
    df = _build_universal_features(df, config)

    # Remove daytime zero-power rows
    if "DC_POWER" in df.columns and irr_col and irr_col in df.columns:
        zero_mask = (df["DC_POWER"] == 0) & (df[irr_col] > 50)
        if zero_mask.sum() > 0:
            df = df[~zero_mask].copy()
            print(f"  Zero-power daytime drop: removed {zero_mask.sum():,} rows")

    # Outlier removal (1-99 percentile on NORM_DC_POWER)
    if "NORM_DC_POWER" in df.columns and df["NORM_DC_POWER"].sum() > 0:
        Q1 = df["NORM_DC_POWER"].quantile(0.01)
        Q3 = df["NORM_DC_POWER"].quantile(0.99)
        mask = (df["NORM_DC_POWER"] >= Q1) & (df["NORM_DC_POWER"] <= Q3)
        removed = (~mask).sum()
        df = df[mask].copy()
        print(f"  Outlier removal: removed {removed:,} rows")

    df = df.fillna(0).reset_index(drop=True)

    # ── 27 Universal Feature Columns ─────────────────────────
    feature_cols = [
        # Normalized power & electrical (capacity-agnostic)
        "NORM_DC_POWER", "NORM_AC_POWER",
        "NORM_DC_VOLTAGE", "NORM_DC_CURRENT",
        "NORM_POWER_VOLATILITY", "NORM_CURRENT_IMBALANCE", "NORM_VOLTAGE_SPREAD",
        # Ratio & efficiency features (already universal)
        "DC_AC_RATIO", "EFFICIENCY", "PERFORMANCE_RATIO", "DC_AC_DEV",
        # Temperature (universal)
        "AMBIENT_TEMP", "MODULE_TEMP", "TEMP_COEFF",
        "TEMP_DIFFERENCE", "TEMP_DEV", "WIND_SPEED",
        # Irradiance & sky (universal)
        "IRRADIATION", "CLEARNESS_INDEX", "DIFFUSE_RATIO",
        "CLOUD_COVER_EST", "SKY_TEMP_EST",
        # Solar geometry (universal)
        "ZENITH_ANGLE", "AZIMUTH_ANGLE",
        # Time (universal)
        "HOUR", "DAY_OF_YEAR", "MONTH",
        # AC current fraction (universal relative)
        "DC_AC_RATIO",   # kept from above list
    ]

    # Deduplicate while preserving order
    seen = set()
    feature_cols_dedup = []
    for f in feature_cols:
        if f not in seen:
            seen.add(f)
            feature_cols_dedup.append(f)
    feature_cols = feature_cols_dedup[:29]   # exactly 29

    missing = [f for f in feature_cols if f not in df.columns]
    if missing:
        print(f"  [WARN] Missing features set to 0: {missing}")
        for f in missing:
            df[f] = 0

    useful = sum(1 for f in feature_cols if df[f].std() > 0)
    print(f"  Data quality: {useful}/{len(feature_cols)} features with variance > 0")
    print(f"  Final: {len(df):,} rows, {len(feature_cols)} features")

    return df, feature_cols


df_processed, FEATURE_COLS = preprocess_and_engineer_universal(df_raw, CONFIG)
processed_save_path = os.path.join(CONFIG["DATA_DIR"], "processed_dataset_universal.csv")
df_processed.to_csv(processed_save_path, index=False)
print(f"Saved: {processed_save_path}  ({os.path.getsize(processed_save_path)/1e6:.1f} MB)")
print(f"Feature columns: {FEATURE_COLS}")


  Starting with 175,320 rows
  Night-drop: removed 90,621 rows (POA_IRRADIANCE < 50 W/m²)
  Engineering universal features...
    [FIX] IRRADIATION mapped from POA_IRRADIANCE
    [OK] MODULE_TEMP via Faiman model: mean=31.7°C
  Zero-power daytime drop: removed 2,106 rows
  Outlier removal: removed 1,652 rows
  Data quality: 27/27 features with variance > 0
  Final: 80,941 rows, 27 features
Saved: /content/drive/MyDrive/SolarGard4/solar_data/processed_dataset_universal.csv  (139.4 MB)
Feature columns: ['NORM_DC_POWER', 'NORM_AC_POWER', 'NORM_DC_VOLTAGE', 'NORM_DC_CURRENT', 'NORM_POWER_VOLATILITY', 'NORM_CURRENT_IMBALANCE', 'NORM_VOLTAGE_SPREAD', 'DC_AC_RATIO', 'EFFICIENCY', 'PERFORMANCE_RATIO', 'DC_AC_DEV', 'AMBIENT_TEMP', 'MODULE_TEMP', 'TEMP_COEFF', 'TEMP_DIFFERENCE', 'TEMP_DEV', 'WIND_SPEED', 'IRRADIATION', 'CLEARNESS_INDEX', 'DIFFUSE_RATIO', 'CLOUD_COVER_EST', 'SKY_TEMP_EST', 'ZENITH_ANGLE', 'AZIMUTH_ANGLE', 'HOUR', 'DAY_OF_YEAR', 'MONTH']


## Step 5 — Inject Faults with PVLib (Physical Simulation)

In [9]:
def setup_pvlib_system(config):
    location = Location(
        latitude=39.01, longitude=-122.19,
        tz="America/Los_Angeles", altitude=30,
        name=f"PVDAQ_{config['STATION_ID']}"
    )
    module_params   = {"pdc0": config["DC_CAPACITY_W"], "gamma_pdc": config["TEMP_COEFF_POWER"]}
    inverter_params = {"pdc0": config["DC_CAPACITY_W"] * config["INVERTER_EFF"],
                       "eta_inv_nom": config["INVERTER_EFF"]}
    mount  = FixedMount(surface_tilt=20, surface_azimuth=180)
    array  = Array(mount=mount, module_parameters=module_params,
                   modules_per_string=12, strings=config["NUM_INVERTERS"])
    pv_sys = PVSystem(arrays=[array], inverter_parameters=inverter_params)
    print(f"PVLib system: {location.name} ({location.latitude}°N, {location.longitude}°E)")
    return {"location": location, "pv_system": pv_sys,
            "module_params": module_params, "inverter_params": inverter_params}

pvlib_system = setup_pvlib_system(CONFIG)


PVLib system: PVDAQ_2107 (39.01°N, -122.19°E)


In [10]:
class PVLibFaultInjector:
    """Inject physically-realistic faults using PVLib models.
    Strategy: compute fault/normal ratio via PVWatts, apply to real data.
    After injection, ALL absolute features are re-normalized so the
    universal feature set remains consistent.
    """

    def __init__(self, config, pvlib_system):
        self.dc_cap    = config["DC_CAPACITY_W"]
        self.ac_cap    = config["DC_CAPACITY_W"] * config["INVERTER_EFF"]
        self.inv_eff   = config["INVERTER_EFF"]
        self.temp_coeff= config["TEMP_COEFF_POWER"]
        self.ref_temp  = config["REF_TEMP"]
        self.num_inv   = config["NUM_INVERTERS"]
        self.fault_labels = config["FAULT_LABELS"]
        self.rated_dc_v   = self.num_inv * 480
        self.rated_dc_i   = self.dc_cap / self.rated_dc_v

    def _pvwatts_dc(self, g, t, pdc0):
        return pvlib.pvsystem.pvwatts_dc(g, t, pdc0, self.temp_coeff, self.ref_temp)

    def _pvwatts_ac(self, pdc, pdc0, eta):
        return pvlib.inverter.pvwatts(pdc, pdc0, eta_inv_nom=eta)

    def _renormalize(self, df, indices):
        """Recalculate ALL normalized features after fault injection."""
        dc_p = df.loc[indices, "DC_POWER"].values
        ac_p = df.loc[indices, "AC_POWER"].values
        irr  = df.loc[indices, "IRRADIATION"].values

        # Normalized power
        df.loc[indices, "NORM_DC_POWER"] = np.clip(dc_p / self.dc_cap, 0, 1.5)
        df.loc[indices, "NORM_AC_POWER"] = np.clip(ac_p / self.ac_cap, 0, 1.5)

        # Ratio features
        df.loc[indices, "DC_AC_RATIO"]       = np.where(ac_p > 0, dc_p / ac_p, 0)
        df.loc[indices, "EFFICIENCY"]        = np.where(irr > 0, (dc_p / (irr * self.dc_cap / 1000)) * 100, 0)
        df.loc[indices, "PERFORMANCE_RATIO"] = np.where(irr > 0, (ac_p / (irr / 1000 * self.dc_cap * self.inv_eff)) * 100, 0)
        df.loc[indices, "DC_AC_DEV"]         = df.loc[indices, "DC_AC_RATIO"].values - (1 / self.inv_eff)

        # DC current & normalized
        dc_v = df.loc[indices, "NORM_DC_VOLTAGE"].values * self.rated_dc_v
        dc_i = np.where(dc_v > 0, dc_p / dc_v, 0)
        df.loc[indices, "NORM_DC_CURRENT"] = np.clip(dc_i / (self.rated_dc_i + 1e-6), 0, 2.0)

    def inject_partial_shading(self, df, indices, severities):
        irr     = df.loc[indices, "IRRADIATION"].values
        temp    = df.loc[indices, "MODULE_TEMP"].values
        real_dc = df.loc[indices, "DC_POWER"].values
        real_ac = df.loc[indices, "AC_POWER"].values
        shaded_frac  = severities * 0.5
        shade_factor = 1 - severities * 0.6
        dc_unshed = self._pvwatts_dc(irr, temp, self.dc_cap * (1 - shaded_frac))
        dc_shed   = self._pvwatts_dc(irr * shade_factor, temp, self.dc_cap * shaded_frac)
        dc_faulted= np.maximum(dc_unshed + dc_shed, 0)
        dc_normal = self._pvwatts_dc(irr, temp, self.dc_cap)
        ratio     = np.where(dc_normal > 0, dc_faulted / dc_normal, 0)
        df.loc[indices, "DC_POWER"] = real_dc * ratio
        ac_normal = self._pvwatts_ac(dc_normal, self.dc_cap, self.inv_eff)
        ac_faulted= self._pvwatts_ac(dc_faulted, self.dc_cap, self.inv_eff)
        ac_ratio  = np.where(ac_normal > 0, ac_faulted / ac_normal, 0)
        df.loc[indices, "AC_POWER"] = real_ac * ac_ratio
        imb = np.abs(dc_unshed - dc_shed) / (dc_faulted + 1) * 0.5
        df.loc[indices, "NORM_CURRENT_IMBALANCE"] = np.clip(imb / (self.rated_dc_i + 1e-6), 0, 1)
        self._renormalize(df, indices)
        df.loc[indices, "fault_label"] = 1

    def inject_soiling(self, df, indices, severities):
        irr     = df.loc[indices, "IRRADIATION"].values
        temp    = df.loc[indices, "MODULE_TEMP"].values
        real_dc = df.loc[indices, "DC_POWER"].values
        real_ac = df.loc[indices, "AC_POWER"].values
        soiling = 1 - severities * 0.25
        dc_normal  = self._pvwatts_dc(irr, temp, self.dc_cap)
        dc_faulted = self._pvwatts_dc(irr * soiling, temp, self.dc_cap)
        ratio  = np.where(dc_normal > 0, dc_faulted / dc_normal, 0)
        df.loc[indices, "DC_POWER"] = real_dc * ratio
        ac_normal  = self._pvwatts_ac(dc_normal, self.dc_cap, self.inv_eff)
        ac_faulted = self._pvwatts_ac(dc_faulted, self.dc_cap, self.inv_eff)
        ac_ratio = np.where(ac_normal > 0, ac_faulted / ac_normal, 0)
        df.loc[indices, "AC_POWER"] = real_ac * ac_ratio
        self._renormalize(df, indices)
        df.loc[indices, "fault_label"] = 2

    def inject_degradation(self, df, indices, severities):
        irr     = df.loc[indices, "IRRADIATION"].values
        temp    = df.loc[indices, "MODULE_TEMP"].values
        real_dc = df.loc[indices, "DC_POWER"].values
        real_ac = df.loc[indices, "AC_POWER"].values
        pdc0_deg  = self.dc_cap * (1 - severities * 0.08)
        dc_normal = self._pvwatts_dc(irr, temp, self.dc_cap)
        dc_fault  = self._pvwatts_dc(irr, temp, pdc0_deg)
        ratio     = np.where(dc_normal > 0, dc_fault / dc_normal, 0)
        df.loc[indices, "DC_POWER"] = real_dc * ratio
        ac_normal = self._pvwatts_ac(dc_normal, self.dc_cap, self.inv_eff)
        ac_fault  = self._pvwatts_ac(dc_fault, self.dc_cap, self.inv_eff)
        ac_ratio  = np.where(ac_normal > 0, ac_fault / ac_normal, 0)
        df.loc[indices, "AC_POWER"] = real_ac * ac_ratio
        # Degradation: slight voltage drop
        df.loc[indices, "NORM_DC_VOLTAGE"] *= (1 - severities * 0.03)
        self._renormalize(df, indices)
        df.loc[indices, "fault_label"] = 3

    def inject_inverter_fault(self, df, indices, severities):
        irr    = df.loc[indices, "IRRADIATION"].values
        temp   = df.loc[indices, "MODULE_TEMP"].values
        real_ac= df.loc[indices, "AC_POWER"].values
        dc_normal  = self._pvwatts_dc(irr, temp, self.dc_cap)
        eta_fault  = self.inv_eff * (1 - severities * 0.8)
        ac_normal  = self._pvwatts_ac(dc_normal, self.dc_cap, self.inv_eff)
        ac_faulted = self._pvwatts_ac(dc_normal, self.dc_cap, eta_fault)
        ac_ratio   = np.where(ac_normal > 0, ac_faulted / ac_normal, 0)
        df.loc[indices, "AC_POWER"] = real_ac * ac_ratio
        self._renormalize(df, indices)
        df.loc[indices, "fault_label"] = 4

    def inject_open_circuit(self, df, indices, severities):
        irr     = df.loc[indices, "IRRADIATION"].values
        temp    = df.loc[indices, "MODULE_TEMP"].values
        real_dc = df.loc[indices, "DC_POWER"].values
        real_ac = df.loc[indices, "AC_POWER"].values
        strings_lost    = np.maximum(1, (severities * 4).astype(int))
        remaining_frac  = 1 - (strings_lost / 7)
        pdc0_reduced    = self.dc_cap * remaining_frac
        dc_normal  = self._pvwatts_dc(irr, temp, self.dc_cap)
        dc_faulted = self._pvwatts_dc(irr, temp, pdc0_reduced)
        ratio      = np.where(dc_normal > 0, dc_faulted / dc_normal, 0)
        df.loc[indices, "DC_POWER"] = real_dc * ratio
        ac_normal  = self._pvwatts_ac(dc_normal, self.dc_cap, self.inv_eff)
        ac_faulted = self._pvwatts_ac(dc_faulted, self.dc_cap, self.inv_eff)
        ac_ratio   = np.where(ac_normal > 0, ac_faulted / ac_normal, 0)
        df.loc[indices, "AC_POWER"] = real_ac * ac_ratio
        df.loc[indices, "NORM_DC_CURRENT"] *= remaining_frac
        self._renormalize(df, indices)
        df.loc[indices, "fault_label"] = 5

    def inject_short_circuit(self, df, indices, severities):
        irr     = df.loc[indices, "IRRADIATION"].values
        temp    = df.loc[indices, "MODULE_TEMP"].values
        real_dc = df.loc[indices, "DC_POWER"].values
        real_ac = df.loc[indices, "AC_POWER"].values
        irr_factor   = 1 - severities * 0.5
        temp_elevated= temp + severities * 30
        dc_normal    = self._pvwatts_dc(irr, temp, self.dc_cap)
        dc_faulted   = self._pvwatts_dc(irr * irr_factor, temp_elevated, self.dc_cap)
        ratio        = np.where(dc_normal > 0, dc_faulted / dc_normal, 0)
        df.loc[indices, "DC_POWER"] = real_dc * ratio
        ac_normal    = self._pvwatts_ac(dc_normal, self.dc_cap, self.inv_eff)
        ac_faulted   = self._pvwatts_ac(dc_faulted, self.dc_cap, self.inv_eff)
        ac_ratio     = np.where(ac_normal > 0, ac_faulted / ac_normal, 0)
        df.loc[indices, "AC_POWER"] = real_ac * ac_ratio
        df.loc[indices, "NORM_DC_VOLTAGE"] *= (1 - severities * 0.4)
        df.loc[indices, "MODULE_TEMP"]      = temp_elevated
        df.loc[indices, "NORM_VOLTAGE_SPREAD"] = severities * 0.3
        self._renormalize(df, indices)
        df.loc[indices, "fault_label"] = 6

    def inject_sensor_fault(self, df, indices, severities):
        n          = len(indices)
        fault_kinds= np.random.random(n)
        irr        = df.loc[indices, "IRRADIATION"].values.astype(np.float64)
        mask_stuck = fault_kinds < 0.33
        irr[mask_stuck]  = np.random.uniform(200, 600, mask_stuck.sum())
        mask_offset= (fault_kinds >= 0.33) & (fault_kinds < 0.66)
        irr[mask_offset] += np.random.uniform(-200, 200, mask_offset.sum())
        mask_spike = fault_kinds >= 0.66
        irr[mask_spike]  *= np.random.uniform(1.5, 3.0, mask_spike.sum())
        irr = np.maximum(irr, 0)
        df.loc[indices, "IRRADIATION"] = irr
        dc_p = df.loc[indices, "DC_POWER"].values
        ac_p = df.loc[indices, "AC_POWER"].values
        df.loc[indices, "EFFICIENCY"]        = np.where(irr > 0, (dc_p / (irr * self.dc_cap / 1000)) * 100, 0)
        df.loc[indices, "PERFORMANCE_RATIO"] = np.where(irr > 0, (ac_p / (irr / 1000 * self.dc_cap * self.inv_eff)) * 100, 0)
        df.loc[indices, "CLEARNESS_INDEX"]   = np.clip(
            irr / (1000 * np.cos(df.loc[indices, "ZENITH_ANGLE"].values * np.pi / 180) + 0.01), 0, 1.5)
        df.loc[indices, "DIFFUSE_RATIO"]     = np.clip(1 - df.loc[indices, "CLEARNESS_INDEX"].values * 0.8, 0, 1)
        df.loc[indices, "CLOUD_COVER_EST"]   = np.clip((1 - df.loc[indices, "CLEARNESS_INDEX"].values) * 100, 0, 100)
        # Corrupt temperature sensors (50% of rows)
        corrupt_mask = np.random.random(n) < 0.5
        ambient_t = df.loc[indices, "AMBIENT_TEMP"].values.astype(np.float64)
        module_t  = df.loc[indices, "MODULE_TEMP"].values.astype(np.float64)
        stuck = corrupt_mask & (np.random.random(n) < 0.5)
        ambient_t[stuck] = 25.0
        module_t[stuck]  = 45.0
        offset = corrupt_mask & ~stuck
        ambient_t[offset] += np.random.uniform(-10, 10, offset.sum())
        module_t[offset]  += np.random.uniform(-15, 15, offset.sum())
        df.loc[indices, "AMBIENT_TEMP"]    = ambient_t
        df.loc[indices, "MODULE_TEMP"]     = module_t
        df.loc[indices, "TEMP_DIFFERENCE"] = module_t - ambient_t
        df.loc[indices, "TEMP_DEV"]        = module_t - (ambient_t + (irr / 800) * 30)
        df.loc[indices, "TEMP_COEFF"]      = 1 + self.temp_coeff * (module_t - self.ref_temp)
        df.loc[indices, "fault_label"] = 7

    def inject_all_faults(self, df, fault_ratio=0.875):
        df = df.copy()
        df["fault_label"] = 0
        df["fault_type"]  = "Normal"
        n_total     = len(df)
        n_faults    = int(n_total * fault_ratio)
        fault_types = [1, 2, 3, 4, 5, 6, 7]
        n_per_fault = n_faults // len(fault_types)
        all_indices = np.random.permutation(n_total)
        fault_idx   = 0
        injectors   = {
            1: self.inject_partial_shading, 2: self.inject_soiling,
            3: self.inject_degradation,     4: self.inject_inverter_fault,
            5: self.inject_open_circuit,    6: self.inject_short_circuit,
            7: self.inject_sensor_fault,
        }
        for ftype in fault_types:
            idx_array  = all_indices[fault_idx:fault_idx + n_per_fault]
            severities = np.random.uniform(0.5, 1.0, n_per_fault)
            injectors[ftype](df, idx_array, severities)
            fault_idx += n_per_fault
        df["fault_type"] = df["fault_label"].map(self.fault_labels)
        return df


def inject_faults(df, config, pvlib_system):
    print("Injecting faults using PVLib physical simulation (Universal version)...")
    injector   = PVLibFaultInjector(config, pvlib_system)
    df_faulted = injector.inject_all_faults(df, fault_ratio=0.875)
    print("\nFault Distribution:")
    print("-" * 40)
    dist = df_faulted["fault_label"].value_counts().sort_index()
    for label, count in dist.items():
        name = config["FAULT_LABELS"].get(label, f"F{label}")
        pct  = count / len(df_faulted) * 100
        print(f"  F{label} ({name:20s}): {count:6,} ({pct:5.1f}%)")
    print("-" * 40)
    print(f"  Total: {len(df_faulted):,} rows")
    return df_faulted


df_faulted = inject_faults(df_processed, CONFIG, pvlib_system)

faulted_save_path = os.path.join(CONFIG["DATA_DIR"], "faulted_dataset_universal.csv")
df_faulted.to_csv(faulted_save_path, index=False)
print(f"\nSaved universal faulted dataset: {faulted_save_path}")
print(f"Size: {os.path.getsize(faulted_save_path) / 1e6:.1f} MB")


Injecting faults using PVLib physical simulation (Universal version)...

Fault Distribution:
----------------------------------------
  F0 (Normal              ): 10,122 ( 12.5%)
  F1 (Partial Shading     ): 10,117 ( 12.5%)
  F2 (Soiling             ): 10,117 ( 12.5%)
  F3 (Degradation         ): 10,117 ( 12.5%)
  F4 (Inverter Fault      ): 10,117 ( 12.5%)
  F5 (Open-Circuit String ): 10,117 ( 12.5%)
  F6 (Short-Circuit       ): 10,117 ( 12.5%)
  F7 (Sensor Fault        ): 10,117 ( 12.5%)
----------------------------------------
  Total: 80,941 rows

Saved universal faulted dataset: /content/drive/MyDrive/SolarGard4/solar_data/faulted_dataset_universal.csv
Size: 141.0 MB


## Step 6 — Data Quality Verification

In [11]:
print("=" * 70)
print("  UNIVERSAL FEATURE QUALITY VERIFICATION")
print("=" * 70)

for feat in FEATURE_COLS:
    if feat in df_faulted.columns:
        std_val  = df_faulted[feat].std()
        mean_val = df_faulted[feat].mean()
        zero_pct = (df_faulted[feat] == 0).sum() / len(df_faulted) * 100
        icon     = "V" if std_val > 0 else "X"
        print(f"  [{icon}] {feat:<28s}: mean={mean_val:>10.4f}, std={std_val:>8.4f}, zeros={zero_pct:>5.1f}%")

useful = sum(1 for f in FEATURE_COLS if f in df_faulted.columns and df_faulted[f].std() > 0)
print(f"\n  Features with variance > 0 : {useful}/{len(FEATURE_COLS)}")
print(f"  Data Quality Score         : {useful/len(FEATURE_COLS)*100:.0f}/100")

print("\nPer-fault mean values (normalized features):")
key_feats = ["NORM_DC_POWER", "NORM_AC_POWER", "EFFICIENCY", "PERFORMANCE_RATIO",
             "NORM_DC_VOLTAGE", "NORM_DC_CURRENT"]
header = "  " + "Fault".ljust(8)
for f in key_feats:
    header += f" {f:>20}"
print(header)
print("  " + "-" * (8 + 21 * len(key_feats)))
for fault in range(8):
    subset = df_faulted[df_faulted["fault_label"] == fault]
    row = f"  F{fault:<7}"
    for f in key_feats:
        row += f" {subset[f].mean():>20.4f}"
    print(row)

print("\n" + "=" * 70)
print("  PIPELINE COMPLETE — Files saved:")
print(f"  - processed_dataset_universal.csv")
print(f"  - faulted_dataset_universal.csv  <-- USE THIS FOR TRAINING")
print("=" * 70)


  UNIVERSAL FEATURE QUALITY VERIFICATION
  [V] NORM_DC_POWER               : mean=    0.3714, std=  0.2195, zeros=  0.0%
  [V] NORM_AC_POWER               : mean=    0.3453, std=  0.2197, zeros=  0.0%
  [V] NORM_DC_VOLTAGE             : mean=    0.0548, std=  0.0082, zeros=  0.0%
  [V] NORM_DC_CURRENT             : mean=    1.8884, std=  0.3080, zeros=  0.0%
  [V] NORM_POWER_VOLATILITY       : mean=    0.0547, std=  0.0736, zeros=  0.0%
  [V] NORM_CURRENT_IMBALANCE      : mean=    0.6943, std=  0.3981, zeros=  0.0%
  [V] NORM_VOLTAGE_SPREAD         : mean=    0.0293, std=  0.0755, zeros=  0.0%
  [V] DC_AC_RATIO                 : mean=    1.2566, std=  0.6768, zeros=  0.0%
  [V] EFFICIENCY                  : mean=   74.4641, std=404.0596, zeros=  0.2%
  [V] PERFORMANCE_RATIO           : mean=   68.8742, std=409.1350, zeros=  0.2%
  [V] DC_AC_DEV                   : mean=    0.2150, std=  0.6768, zeros=  0.0%
  [V] AMBIENT_TEMP                : mean=   22.1456, std=  8.0953, zeros=  0.1%